# DVF: Filter Houses in Ensues (13820)
# Author: Xavier VAN AUSLOOS
# Date: 19/02/26
# Notebook for filtering houses located in Ensues (postal code 13820) from the full France dataset
# Dataset: df_grouped_2020_2025_france_cleaned.csv
# Output: df_2020_2025_houses_ensues.csv

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Ensure project src is on path
_project_root = Path.cwd() if (Path.cwd() / "src" / "dvf").exists() else Path.cwd().parent
_src = _project_root / "src"
if _src.exists() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

In [9]:
# Define paths
input_path = _project_root / "data" / "processed" / "df_grouped_2020_2025_france_cleaned.csv"
output_path = _project_root / "data" / "processed" / "df_2020_2025_houses_ensues.csv"

print(f"Input file: {input_path}")
print(f"Output file: {output_path}")
print(f"Input file exists: {input_path.exists()}")

Input file: /Users/xaviervanausloos/projects/ldi_dvf/data/processed/df_grouped_2020_2025_france_cleaned.csv
Output file: /Users/xaviervanausloos/projects/ldi_dvf/data/processed/df_2020_2025_houses_ensues.csv
Input file exists: True


In [10]:
# Load the full dataset
print("Loading dataset...")
df = pd.read_csv(input_path, low_memory=False)
print(f"Loaded {len(df):,} rows")
print(f"Columns: {len(df.columns)}")
print(f"\nFirst few columns: {list(df.columns[:10])}")

Loading dataset...
Loaded 2,502,263 rows
Columns: 42

First few columns: ['Identifiant de document', 'Reference document', '1 Articles CGI', '2 Articles CGI', '3 Articles CGI', '4 Articles CGI', '5 Articles CGI', 'No disposition', 'Nature mutation', 'No voie']


In [11]:
# Check data types and unique values for filtering columns
print("Checking 'Type local' column:")
print(df['Type local'].value_counts() if 'Type local' in df.columns else "Column not found")

print("\nChecking 'Code postal' column:")
print(f"Data type: {df['Code postal'].dtype if 'Code postal' in df.columns else 'Column not found'}")
print(f"Unique postal codes (sample): {df['Code postal'].unique()[:20] if 'Code postal' in df.columns else 'Column not found'}")

# Check for Ensues postal code in different formats
if 'Code postal' in df.columns:
    # Try as float/int
    print(f"\nPostal code 13820 as float: {(df['Code postal'] == 13820.0).sum()} rows")
    print(f"Postal code 13820 as int: {(df['Code postal'] == 13820).sum()} rows")
    # Check if there are any codes starting with 13820
    postal_str = df['Code postal'].astype(str)
    print(f"Postal codes containing '13820': {(postal_str.str.contains('13820', na=False)).sum()} rows")
    print(f"Sample of postal codes with 13820: {postal_str[postal_str.str.contains('13820', na=False)].unique()[:10]}")
    
    # Check Commune column for Ensues
    if 'Commune' in df.columns:
        print(f"\nChecking 'Commune' column for Ensues:")
        print(f"Rows with 'Ensues' in Commune: {df['Commune'].astype(str).str.contains('Ensues', case=False, na=False).sum()}")
        print(f"Unique communes containing 'Ensues': {df[df['Commune'].astype(str).str.contains('Ensues', case=False, na=False)]['Commune'].unique()}")

Checking 'Type local' column:
Type local
Maison    2502263
Name: count, dtype: int64

Checking 'Code postal' column:
Data type: float64
Unique postal codes (sample): [1000. 1090. 1100. 1110. 1120. 1130. 1140. 1150. 1160. 1170. 1190. 1200.
 1210. 1220. 1230. 1240. 1250. 1260. 1270. 1280.]

Postal code 13820 as float: 253 rows
Postal code 13820 as int: 253 rows
Postal codes containing '13820': 253 rows
Sample of postal codes with 13820: ['13820.0']

Checking 'Commune' column for Ensues:
Rows with 'Ensues' in Commune: 253
Unique communes containing 'Ensues': ['ENSUES-LA-REDONNE']


In [12]:
# Filter for houses in Ensues (postal code 13820)
print("Filtering data...")
print(f"Initial row count: {len(df):,}")

# Filter by Type local = "Maison"
df_houses = df[df['Type local'] == 'Maison'].copy()
print(f"After filtering houses: {len(df_houses):,} rows")

# Filter by Code postal = 13820 (Ensues)
# Handle both float and int formats - convert to numeric first, then compare
# The postal code is stored as float (13820.0), so we need to handle that
df_houses['Code postal'] = pd.to_numeric(df_houses['Code postal'], errors='coerce')
df_ensues = df_houses[df_houses['Code postal'] == 13820].copy()
print(f"After filtering Ensues (postal code 13820): {len(df_ensues):,} rows")

# Alternative: also try filtering by Commune name if postal code doesn't work
if len(df_ensues) == 0:
    print("\n⚠️  No rows found with postal code 13820. Trying alternative: filtering by Commune name...")
    df_houses_alt = df[df['Type local'] == 'Maison'].copy()
    df_ensues = df_houses_alt[df_houses_alt['Commune'].astype(str).str.contains('Ensues', case=False, na=False)].copy()
    print(f"After filtering by Commune name 'Ensues': {len(df_ensues):,} rows")
    if len(df_ensues) > 0:
        print(f"Postal codes in filtered data: {df_ensues['Code postal'].unique()}")

print(f"\nFiltered dataset represents {len(df_ensues)/len(df)*100:.2f}% of original dataset")

Filtering data...
Initial row count: 2,502,263
After filtering houses: 2,502,263 rows
After filtering Ensues (postal code 13820): 253 rows

Filtered dataset represents 0.01% of original dataset


In [13]:
# Verify the filtering
print("Verification:")
print(f"All are houses: {(df_ensues['Type local'] == 'Maison').all()}")
print(f"All are in Ensues: {(df_ensues['Code postal'] == '13820').all()}")
print(f"\nCommune values: {df_ensues['Commune'].unique() if 'Commune' in df_ensues.columns else 'N/A'}")
print(f"\nSample of filtered data:")
df_ensues[['Code postal', 'Commune', 'Type local', 'Surface reelle bati', 'Voie']].head()

Verification:
All are houses: True
All are in Ensues: False

Commune values: ['ENSUES-LA-REDONNE']

Sample of filtered data:


,Code postal,Commune,Type local,Surface reelle bati,Voie
242344,13820.0,ENSUES-LA-REDONNE,Maison,80.0,VAL DE RICARD
242345,13820.0,ENSUES-LA-REDONNE,Maison,124.0,VAL DE RICARD
242346,13820.0,ENSUES-LA-REDONNE,Maison,79.0,VAL DE RICARD
242347,13820.0,ENSUES-LA-REDONNE,Maison,82.0,VAL DE RICARD
242348,13820.0,ENSUES-LA-REDONNE,Maison,105.0,VAL DE RICARD


In [14]:
# Save filtered dataset to CSV
print(f"Saving filtered dataset to {output_path}...")
df_ensues.to_csv(output_path, index=False)
print(f"✅ Successfully saved {len(df_ensues):,} rows to {output_path}")

# Display summary statistics
print(f"\n📊 Summary:")
print(f"   Total properties in Ensues: {len(df_ensues):,}")
print(f"   Columns: {len(df_ensues.columns)}")
print(f"   File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

Saving filtered dataset to /Users/xaviervanausloos/projects/ldi_dvf/data/processed/df_2020_2025_houses_ensues.csv...
✅ Successfully saved 253 rows to /Users/xaviervanausloos/projects/ldi_dvf/data/processed/df_2020_2025_houses_ensues.csv

📊 Summary:
   Total properties in Ensues: 253
   Columns: 42
   File size: 0.04 MB
